# Ce qu'il reste à faire 

* [ ] Ajouter une variable year à la classe AECS pour sélectionne la période
* [ ] Nettoyer la classe
* [ ] Repasser sur les noms de variable (voir ce qui est factorisable
* [ ] A la toute fin, réaliser une docstring

In [77]:
import string
import pandas as pd
import numpy as np
import seaborn as sns
import re
from datetime import datetime as dt
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [78]:
alphabet = list(string.ascii_lowercase)

In [253]:
class AECS():
    def __init__(self,set_sll_columns=None):      
        sheets2keep = ['Baptiste',
                       'Bruno',
                       'Céline',
                       'Esther',
                       'Gwen',
                       'Hélène',
                       'Inès',
                       'Laetitia C',
                       'Laetitia D',
                       'Mathilde M',
                       'Pascale',
                       'Zina'
                      ]
        
        data_date = '20250917'
        filepath_data = f'../data/aecs/tableaux_suivi_AECS_{data_date}.xlsx'
        
        self.aecs = pd.read_excel(filepath_data,sheet_name=sheets2keep)
        self.aecs = pd.concat(self.aecs)
        self.aecs = self.aecs.droplevel(level=1).reset_index()        
        self.aecs.loc[self.aecs['Nb adultes touchés']>100,'ANOMALIE Nb adultes touchés']='oui'
        self.aecs.loc[self.aecs['Nb enfants touchés']>100,'ANOMALIE Nb enfants touchés']='oui'
        
        self.aecs['Nb actions'] = 1
        
        # Pour toutes les feuilles
        self.aecs['Total personnes distinctes touchées'] = self.aecs['Nb enfants touchés'] + self.aecs['Nb adultes touchés']
        
        #Pour Action Culturelle uniquement on copie dans la colonne "Total" les données de la colonne"Nb personnes touchées"
        self.aecs.loc[self.aecs['index']=="Mathilde M","Total personnes distinctes touchées"] = self.aecs['Nb personnes touchées']
        
        self.aecs_collectivites = self.aecs[self.aecs["index"]=='Laetitia D']
        self.effectifs_scolaires = pd.read_excel(filepath_data,sheet_name='Effectifs')
        self.aecs_collectivites = self.aecs_collectivites.merge(self.effectifs_scolaires,left_on="Nom d'équipement",right_on="Nom",how='left')
        self.list_SchoolNames = list(self.aecs_collectivites[self.aecs_collectivites['Type de structure']=="Ecole"]["Nom d'équipement"].unique())
        
        self.ac = self.aecs[self.aecs["index"]=='Mathilde M']
        self.aes = self.aecs[self.aecs["index"]!='Mathilde M']
        

        if set_sll_columns is not None:
            
            # Ajout des colonnes SLL pour le DataFrame Action Culturelle (ac)
            self.ac.loc[self.ac["Type d'action"]=="Exposition",
                        "sll_type_action"] = "H4 - Exposition"

            self.ac.loc[self.ac["Type d'action"].isin(['Conférence','Rencontre','Lecture']),
                        'sll_type_action'] = "H4 - Conférences, rencontres, lectures"

            self.ac.loc[self.ac["Type d'action"].isin(['Concert','Projection']),
                        'sll_type_action'] = "H4 - Concerts, projections"

            self.ac.loc[self.ac["Type d'action"]=="Séance de contes",
                        "sll_type_action"] = "H4 - Séances de contes"

            self.ac.loc[self.ac["Type d'action"].isin(["Club lecture","Atelier d'écriture"]),
                        'sll_type_action'] = "H4 - Clubs de lecteurs, ateliers d'écriture"

            #ac.loc[ac["Type d'action"].isin(["Journée festive","Salon du livre","Festival"]),['sll-type_action']] = "H4 - Fêtes, salons du livre, festivals" 
            #ac.loc[ac["Evénement"].notna(),["sll_type_action"]] = "H4 - Fêtes, salons du livre, festivals"


            self.ac.loc[~self.ac["sll_type_action"].isin(["H4 - Exposition",
                                                          "H4 - Conférences, rencontres, lectures",
                                                          "H4 - Concerts, projections",
                                                          "H4 - Séances de contes",
                                                          "H4 - Clubs de lecteurs, ateliers d'écriture",
                                                          #"H4 - Fêtes, salons du livre, festivals"
                                                         ]),
                        "sll_type_action"] = "H4 - Autres"
            
            self.ac.loc[self.ac["Type de public"].isin(['Enfants','Petite enfance','Adolescents']),'sll_type_public'] = 'a/ Enfants'
            self.ac.loc[~self.ac['Type de public'].isin(['Enfants','Petite enfance','Adolescents']),'sll_type_public'] = 'b/ Tout public'
    
    
            # Création d'un dictionnaire
            dict_TypeDeStructure = {'Ecole':'Écoles',
                                    'Collège':'Collèges',
                                    'Lycée':'Lycées',
                                    'Supérieur':'Supérieur',
                                    'Maison_de_retraite':'Maisons de retraite',
                                    'Centre_social':'Centres sociaux',
                                    'Centre_de_loisirs':'Centres de loisirs',
                                    'Structure_de_la_petite_enfance':'Services de la petite enfance',
                                    'Service_emploi_et_formation':'Services de l\'emploi',
                                    'Equipement_medicosocial':'Équipements médico-sociaux'
                                   }
            
            # Création d'une liste de clés et de valeurs
            keys = dict_TypeDeStructure.keys()
            values = dict_TypeDeStructure.values()
            column_to_parse = 'Type de structure'
            column_to_add = 'H1 - Partenariats avec des institutions'
            
            # Pour chaque paire clé/valeurs, 
            # si Type de structure contient clé
            # Pour nouvelle colonne H1 - Partenariat avec les institutions, utilisé valeurs associée
            for key,value,letter in zip(keys,values,alphabet):
                self.aecs.loc[self.aecs[column_to_parse]==key,column_to_add] = f'{letter}/ {value}'

        
            dict_TypeDePublic = {"Personnes âgées (65 ans et plus)":"Personnes âgées",
                                 "Personnes en situation de handicap":"Personnes en situation de hancidap",
                                 "Jeunes (18-25 ans)":"Jeunes",
                                 "Petite enfance (0-3 ans)":"Petite enfance",
                                 "Personnes en recherche d'emploi":"Personnes en recherche d'emploi",
                                 "Personnes en situation d'illettrisme":"Personnes en situation d'illétrisme",
                                 "Populations allophones":"Population non-francophone",
                                 "Populations en situation d'insertion sociale":"Population en situation d'insertion sociale"
                                }

            keys = dict_TypeDePublic.keys()
            values = dict_TypeDePublic.values()
            for key,value,letter in zip(keys,values,alphabet):
                self.aecs.loc[self.aecs["Type de public"]==key,
                             'H7 - Actions et services à destination de publics à besoins spécifiques'] = f'{letter}/ {value}'

    def get_SLL_H1_PartenariatsAvecDesInstitutions(self):
        
        self.aecs_H1_df = self.aecs.groupby('H1 - Partenariats avec des institutions')['Total personnes distinctes touchées'].sum().to_frame()
        return(self.aecs_H1_df)

        
        
    def get_SLL_H2_PartenariatsAvecDesEquipementsCulturels(self):
        
        self.aecs_H2_df = self.aecs[self.aecs["Type de structure"]=="Centre_culturel"].groupby("Nom d'équipement")['Total personnes distinctes touchées'].sum().to_frame()
        return(self.aecs_H2_df)
    
    
    #def get_SLL_H3_PartenariatsAvecDesAssociations(self):
        
        
        
    
    
    def get_SLL_H4_ActionsAuSeinDelEtablissement(self):
        
        self.aecs_H4XX_df= self.ac.pivot_table(index='sll_type_action',
                                      columns='sll_type_public',
                                      values=['Nb actions','Nb personnes touchées'],
                                      aggfunc=sum,
                                      margins=True,
                                      fill_value=0,
                                      margins_name='Total'
                                     )
        
        return(self.aecs_H4XX_df)
        
    def get_SLL_H5_ActionsHorsDelEtablissement(self,print_result=False):
        
        self.aecs_H5XX_df_HorsLesMurs = self.aecs[(self.aecs['Lieu']=='Hors les murs') &
                                                  (self.aecs['index']!='Laetitia D')]
        
        self.aecs_5XX_NbActionsHorsLesMursMed = self.aecs_H5XX_df_HorsLesMurs['Nb actions'].sum()
        self.aecs_5XX_NbActionsHorsLesMursCol = len(self.aecs_collectivites)
        self.aecs_502_NbActionsHorsLesMurs = self.aecs_5XX_NbActionsHorsLesMursCol + self.aecs_5XX_NbActionsHorsLesMursMed
            
        
        for name in self.list_SchoolNames:
            aecs_collectivites_school = self.aecs_collectivites[self.aecs_collectivites["Nom d'équipement"]==name]
            Total_actions_BCD = len(aecs_collectivites_school[aecs_collectivites_school["Nom action ou projet"]=="BCD"])
            Total_actions_MARM = len(aecs_collectivites_school[aecs_collectivites_school["Nom action ou projet"]=="MARMOTHEQUE"])
            Total_emprunteurs_distincts = aecs_collectivites_school["Nom emprunteur"].nunique()
            self.aecs_collectivites.loc[self.aecs_collectivites["Nom d'équipement"]==name,'Total_actions_BCD']=Total_actions_BCD
            self.aecs_collectivites.loc[self.aecs_collectivites["Nom d'équipement"]==name,'Total_actions_Marmothèque'] = Total_actions_MARM
            self.aecs_collectivites.loc[self.aecs_collectivites["Nom d'équipement"]==name,'Total_emprunteurs_distincts'] = Total_emprunteurs_distincts
            
        effectif_moyen_classe = 25    
        self.aecs_collectivites.loc[self.aecs_collectivites["Total_actions_BCD"]>0,'Nb enfants touchés'] = self.aecs_collectivites['Total Ecole']
        self.aecs_collectivites.loc[self.aecs_collectivites["Total_actions_Marmothèque"]>0,'Nb enfants touchés'] = self.aecs_collectivites['Total Maternelle']
        self.aecs_collectivites.loc[(self.aecs_collectivites["Total_actions_BCD"]==0) &
                                    (self.aecs_collectivites["Total_actions_Marmothèque"]==0),
                                    'Nb enfants touchés'] = self.aecs_collectivites['Total_emprunteurs_distincts'] * effectif_moyen_classe

        # Suppression des noms doublons pour les noms d'école
        self.aecs_collectivites.drop_duplicates(subset="Nom",
                                                inplace=True)
        
        self.aecs_collectivites_schools = self.aecs_collectivites[["index","Nom","Nb enfants touchés"]]
        
        self.aecs_5XX_PopulationToucheeCol = self.aecs_collectivites_schools["Nb enfants touchés"].sum()
        self.aecs_5XX_PopulationToucheeMed = self.aecs_H5XX_df_HorsLesMurs['Total personnes distinctes touchées'].sum()
        self.aecs_503_PopulationTouchee = self.aecs_5XX_NbActionsHorsLesMursMed + self.aecs_5XX_PopulationToucheeCol
        
        line_break = '\n'
        footer = '-'*10
        
        print(f"H5 - ACTIONS HORS DE L'ÉTABLISSEMENT")
        print(footer)
        print(f"H501 - Actions hors de l'établissement : OUI")
        print(f"H502 - Nombre d'actions hors-les-murs : {self.aecs_502_NbActionsHorsLesMurs}")
        print(f"H503 - Population touchée : {self.aecs_503_PopulationTouchee}")
        print(f"H504 - Chiffres transmis par le service de prêts à domicile")
        
        
    def get_SLL_H7_PublicsSpecifiques(self):
        
        self.aecs_7XX_df = self.aecs.groupby('H7 - Actions et services à destination de publics à besoins spécifiques')['Nb actions','Total personnes distinctes touchées'].sum()
        return(self.aecs_7XX_df)
        
    def get_Statistiques(self,
                         print_result=False,
                         filter_on=None,
                         method = None,
                         group_by_what = [],
                         total_on = [],
                         count_actions_numbers=False):
        
        if filter_on == 'ac':
            self.stats = self.df[self.df.index.get_level_values(0)=='Mathilde M'] # On ne garde que le feuille Excel de Mathilde
            
        if filter_on == 'aecs':
            self.stats = self.df[self.df.index.get_level_values(0)!='Mathilde M'] # On exclut la feuille Excel de Mathilde
        
        self.stats['Date début action'] = pd.to_datetime(self.stats['Date début action'],
                                             errors='coerce',
                                             format='%Y-%m-%d %H:%M:%S')
        
        self.stats['annee'] = pd.DatetimeIndex(self.stats['Date début action']).year
        
            
        self.stats = self.stats.groupby(group_by_what)[total_on].sum()
        return(self.stats)
        

In [254]:
aecs = AECS(set_sll_columns=True)

In [255]:
aecs.get_SLL_H1_PartenariatsAvecDesInstitutions()

,Total personnes distinctes touchées
H1 - Partenariats avec des institutions,
a/ Écoles,3966.0
b/ Collèges,1041.0
c/ Lycées,262.0
d/ Supérieur,349.0
e/ Maisons de retraite,83.0
f/ Centres sociaux,645.0
g/ Centres de loisirs,12.0
h/ Services de la petite enfance,72.0
i/ Services de l'emploi,196.0


In [256]:
aecs.get_SLL_H2_PartenariatsAvecDesEquipementsCulturels()

,Total personnes distinctes touchées
Nom d'équipement,
Archives Nationales du Monde du Travail Roubaix,15.0
Autres,38.0
Cie des Vagabondes,45.0
Compagnie de l'Oiseau-Mouche,35.0
Conservatoire de Roubaix,92.0
Librairie Les Quatre Chemins,2.0


In [257]:
aecs.get_SLL_H4_ActionsAuSeinDelEtablissement()

Nb actions                       \
sll_type_public                             a/ Enfants b/ Tout public Total   
sll_type_action                                                               
H4 - Autres                                         45            291   248   
H4 - Clubs de lecteurs, ateliers d'écriture          2             33    29   
H4 - Concerts, projections                          20             31    45   
H4 - Conférences, rencontres, lectures              32            278   289   
H4 - Exposition                                      0             13    11   
H4 - Séances de contes                              14              7    16   
Total                                               99            539   638   

                                            Nb personnes touchées  \
sll_type_public                                        a/ Enfants   
sll_type_action                                                     
H4 - Autres                                                  1189   
H4 - Clubs de lecteurs, ateliers d'écriture                     8   
H4 - Concerts, projections                                   1194   
H4 - Conférences, rencontres, lectures                       1470   
H4 - Exposition                                                 0   
H4 - Séances de contes                                        259   
Total                                                        4120   

                                                                     
sll_type_public                             b/ Tout public    Total  
sll_type_action                                                      
H4 - Autres                                           9987  11176.0  
H4 - Clubs de lecteurs, ateliers d'écriture            374    382.0  
H4 - Concerts, projections                             937   2131.0  
H4 - Conférences, rencontres, lectures                8100   9570.0  
H4 - Exposition                                        630    630.0  
H4 - Séances de contes                                 207    466.0  
Total                                                20235  24355.0

In [ ]:
aecs.get_SLL_H5_ActionsHorsDelEtablissement(print_result=True)

In [240]:
aecs.get_SLL_H7_PublicsSpecifiques()

,Nb actions,Total personnes distinctes touchées
H7 - Actions et services à destination de publics à besoins spécifiques,,
a/ Personnes âgées,8,63.0
b/ Personnes en situation de hancidap,1,16.0
c/ Jeunes,10,197.0
d/ Petite enfance,101,2062.0
g/ Population non-francophone,23,348.0
h/ Population en situation d'insertion sociale,3,27.0


In [ ]:
aecs.aecs_collectivites.columns

In [117]:
aecs.aecs_collectivites[['index',
                         'Date début action',
                         "Nom d'équipement",
                         "Nom",
                         "Nb enfants touchés"]].drop_duplicates(subset="Nom")

,index,Date début action,Nom d'équipement,Nom,Nb enfants touchés
0,Laetitia D,2023-09-05 00:00:00,Jules Guesde,Jules Guesde,375.0
6,Laetitia D,2023-09-05 00:00:00,Léo Lagrange,Léo Lagrange,225.0
8,Laetitia D,2023-09-05 00:00:00,Pierre Brossolette,Pierre Brossolette,175.0
10,Laetitia D,2023-09-05 00:00:00,Léon Jouhaux,Léon Jouhaux,113.0
12,Laetitia D,2023-09-05 00:00:00,Lavoisier,Lavoisier,NaN
21,Laetitia D,2023-09-05 00:00:00,Jean Macé,Jean Macé,231.0
30,Laetitia D,2023-09-05 00:00:00,Marie Auxiliatrice,Marie Auxiliatrice,100.0
34,Laetitia D,2023-09-05 00:00:00,Michelet,Michelet,300.0
44,Laetitia D,2023-09-05 00:00:00,Pierre de Ronsard niv.2,Pierre de Ronsard niv.2,154.0
45,Laetitia D,2023-09-05 00:00:00,Montesquieu,Montesquieu,116.0


In [121]:
aecs.aecs_collectivites.groupby("Nom").size()

Nom
Albert Camus                76
Albert Samain                3
Alphonse Daudet             10
Anatole France              54
Blaise Pascal               43
Boileau / Pasteur           55
Buffon                      10
Charles Perrault             9
Condorcet                   56
Edmond Rostand              20
Edouard Vaillant            21
Elsa Triolet                32
Ernest Legouvé              83
Ernest Legouvé 2            35
Ernest Renan                24
François Villon             48
George Sand                  2
Henri Carette                2
Jacques Prévert              4
Jean Macé                   98
Jeanne d'Arc                49
Jules Ferry                 48
Jules Guesde                87
Jules Verne                 13
La Cordée                   10
Lakanal                     29
Lavoisier                   53
Linné                       24
Littré                      65
Lucie Aubrac                83
Léo Lagrange                14
Léon Gambetta               46
Léon

# Pistes pour dataviz

1. Evolution des actions AC vs AECS
2. AC 
    * Reprendre les vues existantes sur Kibini
    
3. AECS
    * Evolution du nombre d'actions selon 
        - Type d'action
        - Type de public
        - Type d'établissement (colonne = `en partenariat avec`)
    * Evolution de la part de chaque type d'action
    * Evolution du nb de personnes touchées 
        - Type d'action
        - Type de public
        - Type d'établissement
    * Cartographie des actions
        - en général
        - dans les murs vs hors les murs
    
    * Focus sur :
        - Les visites de classes
        - Les dépôts aux écoles
        - Les jeux-vidéos ?